In [1]:
pip install unidecode

In [2]:
import pandas as pd
import cv2
import numpy as np
import requests
import json
from unidecode import unidecode
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# Load the CSV file containing user data into a DataFrame
users_df = pd.read_csv('/content/drive/MyDrive/Thesis/data-extraction/users-df-creation/kaggle_users.csv', index_col=[0])

# Drop rows with missing values (NaN) from the DataFrame in-place
users_df.dropna(inplace=True)

In [4]:
users_df

,Username,Display_Name,Image_Path
0,bestfitting,bestfitting,"""https://storage.googleapis.com/kaggle-avatars..."
1,philippsinger,Psi,"""https://storage.googleapis.com/kaggle-avatars..."
2,christofhenkel,Dieter,"""https://storage.googleapis.com/kaggle-avatars..."
3,cdeotte,Chris Deotte,"""https://storage.googleapis.com/kaggle-avatars..."
4,haqishen,Qishen Ha,"""https://storage.googleapis.com/kaggle-avatars..."
...,...,...,...
3289,phunter,phunter,"""https://storage.googleapis.com/kaggle-avatars..."
3291,ggglhf,tinrtgu,"""https://storage.googleapis.com/kaggle-avatars..."
3292,ngutten,Nicholas Guttenberg,"""https://storage.googleapis.com/kaggle-avatars..."
3295,wildwizard,WhizWilde,"""https://storage.googleapis.com/kaggle-avatars..."


In [5]:
# Create a new column 'First_Name' in the DataFrame by extracting the first name from 'Display_Name'
users_df['First_Name'] = users_df['Display_Name'].str.split(' ').str[0].astype("string")

# Apply the unidecode function to remove accents and special characters from 'First_Name'
users_df['First_Name'] = users_df['First_Name'].apply(unidecode)

In [6]:
# # Create an instance of the gender detector with case insensitivity
# d = gender.Detector(case_sensitive=False)

# # Define a function that uses the gender detector to guess the gender based on the first name
# def guessGenderByFirstName(first_name: str):
#     return d.get_gender(first_name)

In [7]:
# URL+API_KEY for the genderize.io API
API = "https://api.genderize.io/"
API_KEY = "36eaf2aa81b00f822d35dce85aed293b"
MIN_NAME_COUNT = 1000
MIN_NAME_PROBALITY = 0.8

# Function to get gender prediction based on a name
def get_user_gender(name: str):
    # Construct the API request URL with the provided name
    response = requests.get(f"{API}?name={name}&apikey={API_KEY}")

    # Check if the API request was successful (status code 200)
    if response.status_code == 200:
        print("Successfully fetched the data with the provided parameters")
        text = response.json()  # Parse the response JSON data
        return text  # Return the JSON data containing gender prediction
    else:
        print(f"There's a {response.status_code} error with your request")
        return None  # Return None in case of an error

# Function to predict gender based on genderize.io results
def predict_gender(genderize_results: dict):
    # Check if there are more than 1000 samples and a high probability (greater than 0.8)
    if (genderize_results != None and genderize_results['count'] > MIN_NAME_COUNT and genderize_results['probability'] > MIN_NAME_PROBALITY):
        return genderize_results['gender']  # Return the predicted gender
    else:
        return 'none'  # Return 'none' if conditions are not met for a confident prediction

In [8]:
# Applying the get_user_gender function to each row of the 'First_Name' column to get genderize.io
users_df['First_Name-genderize_results'] = users_df['First_Name'].apply(get_user_gender)

Streaming output truncated to the last 5000 lines.
Successfully fetched the data with the provided parameters
Successfully fetched the data with the provided parameters
Successfully fetched the data with the provided parameters
Successfully fetched the data with the provided parameters
Successfully fetched the data with the provided parameters
Successfully fetched the data with the provided parameters
Successfully fetched the data with the provided parameters
Successfully fetched the data with the provided parameters
Successfully fetched the data with the provided parameters
Successfully fetched the data with the provided parameters
Successfully fetched the data with the provided parameters
Successfully fetched the data with the provided parameters
Successfully fetched the data with the provided parameters
Successfully fetched the data with the provided parameters
Successfully fetched the data with the provided parameters
Successfully fetched the data with the provided parameters
Succe

In [9]:
# The new column will contain gender predictions based on genderize.io results
users_df['First_Name_Guess-genderize'] = users_df['First_Name-genderize_results'].apply(predict_gender)

In [10]:
users_df["First_Name_Guess-genderize"].value_counts()

none      8364
male      5546
female     705
Name: First_Name_Guess-genderize, dtype: int64

In [11]:
users_df['First_Name-genderize_results'].apply(lambda x: 'none' if x is None else x['gender']).value_counts()

male      8907
female    1558
Name: First_Name-genderize_results, dtype: int64

In [12]:
users_df.to_csv('users_with_genders_prediction.csv')